# 03. Exact String Matching Baseline & Candidate Generation

**Course:** CSE 276 – CA-1 Academic Project  
**Project:** Material Code Harmonisation Across Public Enterprises  
**Mandatory Academic Objective 3:** *"Report pairwise precision/recall/F1, cluster purity. Baseline: exact string match."*  
**Input Pool:** `data/processed/preprocessed_materials.csv` (6,634 candidate records)  

---

## Notebook Objectives
1. **Environment Setup:** Dynamic configuration supporting Google Colab and local workstation environments.
2. **Data Ingestion & Validation:** Ingest the 6,634 preprocessed candidate pool records and verify schema integrity.
3. **Exact String Matching Baseline:** Identify repeated normalized descriptions, generate deduplicated pairs ($i < j$), and classify intra- vs cross-enterprise pairs.
4. **Cross-Enterprise Retrieval:** Uncover verbatim matches shared across public enterprises (IOCL, Oil India, NTPC).
5. **Lexical Candidate Generation (TF-IDF):** Implement sub-quadratic sparse matrix retrieval using `TfidfVectorizer(ngram_range=(1,2), min_df=2)` and cosine threshold $\tau = 0.35$.
6. **Engineering Compatibility Checks:** Audit candidate pairs against extracted technical parameters (material grade, pressure class, schedule, diameter) to detect specification conflicts.
7. **Ground-Truth Label Audit:** Verify label availability in `data/labelled/` and construct a structured reviewer template without fabricating synthetic metrics.
8. **Export & Diagnostic Visualization:** Save exact matches, candidate pairs, review templates, and diagnostic distribution charts.

> **Academic Integrity Rule:** No fabricated ground-truth labels, synthetic precision/recall metrics, or premature ML training. All statistics reflect actual enterprise procurement data.

---
## 1. Environment & Workspace Setup

Anchors project root, configures `sys.path`, and imports core libraries and project modules.

In [ ]:
# ==============================================================================
# Step 1: Environment & Repository Configuration
# ==============================================================================
import os
import sys
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('[COLAB] Running in Google Colab environment.')
    REPO_NAME = 'CSE-276-PROJECT'
    REPO_URL = 'https://github.com/SHIVA07605/CSE-276-PROJECT.git'
    
    if not os.path.exists(f'/content/{REPO_NAME}'):
        print(f'Cloning repository from {REPO_URL}...')
        os.system(f'git clone {REPO_URL}')
    
    PROJECT_ROOT = Path(f'/content/{REPO_NAME}').resolve()
    os.chdir(PROJECT_ROOT)
    os.system('pip install -q pandas numpy matplotlib scikit-learn scipy')
else:
    print('[LOCAL] Running in Local Workstation environment.')
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir

# Add project root to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project Root: {PROJECT_ROOT}')
print(f'sys.path configured: {str(PROJECT_ROOT) in sys.path}')

# Core scientific libraries
import time
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Project module imports
from src.config import (
    DATA_PROCESSED_DIR,
    DATA_LABELLED_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
)
from src.baseline import exact_match_predict, find_exact_matches
from src.candidate_generation import generate_tfidf_candidates
from src.feature_extraction import check_attribute_conflicts

MATCHING_DIR = RESULTS_DIR / 'matching'
MATCHING_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('[OK] All project modules and directory paths loaded.')

---
## 2. Ingestion & Validation of Preprocessed Candidate Pool

Loads `data/processed/preprocessed_materials.csv` produced by Notebook 02 and validates schema completeness.

In [ ]:
# ==============================================================================
# Step 2: Load and Validate Input Dataset
# ==============================================================================
cand_csv = DATA_PROCESSED_DIR / 'preprocessed_materials.csv'
df = pd.read_csv(cand_csv)

print(f'Records Ingested : {len(df):,}')
print(f'Total Columns    : {len(df.columns)}')
print(f'Unique Corpus IDs: {df["corpus_id"].nunique():,}')

# Validation checks
assert len(df) == 6634, f'Expected 6,634 records, found {len(df)}'
assert df['corpus_id'].nunique() == len(df), 'Duplicate corpus IDs detected!'
assert 'cleaned_description' in df.columns, 'Missing cleaned_description column!'
assert 'organization' in df.columns, 'Missing organization column!'
assert 'item_type_hint' in df.columns, 'Missing item_type_hint column!'

print('\nEnterprise Breakdown in Candidate Pool:')
print(df['organization'].value_counts().to_string())

print('\nTop 8 Commodity Category Hints:')
print(df['item_type_hint'].value_counts().head(8).to_string())

---
## 3. Exact String Matching Baseline

### Baseline Formulation
The exact string matching baseline asserts equivalence if and only if two normalized descriptions match verbatim:
$$f_{\text{baseline}}(r_i, r_j) = \mathbb{I}(\text{norm}(D_i) = \text{norm}(D_j))$$

- **Deduplication:** We generate unique unordered pairs $(i, j)$ with $i < j$.
- **Self-Matches:** Records matching themselves are strictly excluded.
- **Enterprise Relationship:** Each pair is tagged as either `intra-enterprise` (same PSU) or `cross-enterprise` (different PSUs).

In [ ]:
# ==============================================================================
# Step 3: Compute Exact String Matches
# ==============================================================================
t0 = time.time()
df_exact = find_exact_matches(df, desc_col='cleaned_description', id_col='corpus_id', org_col='organization')
exact_elapsed = time.time() - t0

print(f'Exact matching completed in {exact_elapsed:.3f} seconds.')
print(f'Total Exact Match Pairs: {len(df_exact):,}')
print(f'Unique Repeated Descriptions: {df_exact["matching_description"].nunique():,}')

print('\nPair Breakdown by Enterprise Relationship:')
breakdown = df_exact['is_cross_enterprise'].value_counts()
print(f'  Intra-enterprise pairs: {breakdown.get(False, 0):,}')
print(f'  Cross-enterprise pairs: {breakdown.get(True, 0):,}')

# Verify no self-matches
self_matches = (df_exact['corpus_id_1'] == df_exact['corpus_id_2']).sum()
assert self_matches == 0, 'Self-matches detected in baseline!'

---
## 4. Cross-Enterprise Exact Matches Inspection

Identifies verbatim identical descriptions that bridge distinct public sector enterprises. These form high-confidence natural clusters for harmonization.

In [ ]:
# ==============================================================================
# Step 4: Display Cross-Enterprise Exact Matches
# ==============================================================================
cross_exact = df_exact[df_exact['is_cross_enterprise']].copy()
print(f'Discovered {len(cross_exact)} Cross-Enterprise Exact Matches:\n')

display_cols = ['corpus_id_1', 'organization_1', 'corpus_id_2', 'organization_2', 'matching_description']
display_df = cross_exact[display_cols].reset_index(drop=True)
print(display_df.to_string())

---
## 5. Sub-Quadratic Candidate Generation via TF-IDF Vectorization

### Why Blocking is Required
Exhaustive pairwise comparison across 6,634 candidate records requires:
$$\frac{6,634 \times 6,633}{2} = 22,001,211 \text{ pairs}$$
Evaluating 22 million pairs with heavy classifiers or deep transformers is computationally prohibitive ($O(N^2)$).

### Lexical Inverted Indexing
We use `TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)` and chunked sparse matrix multiplication $(\mathbf{X} \cdot \mathbf{X}^T)$ with cosine similarity cutoff $\tau = 0.35$ to prune the search space to tractable, high-recall candidate pairs.

In [ ]:
# ==============================================================================
# Step 5: Sub-Quadratic TF-IDF Candidate Retrieval
# ==============================================================================
SIMILARITY_THRESHOLD = 0.35

t0 = time.time()
df_tfidf = generate_tfidf_candidates(
    df,
    text_col='cleaned_description',
    id_col='corpus_id',
    org_col='organization',
    min_similarity=SIMILARITY_THRESHOLD,
    ngram_range=(1, 2),
    min_df=2,
    batch_size=500,
    block_by_hint=False,
    cross_enterprise_only=False,
)
tfidf_elapsed = time.time() - t0

print(f'TF-IDF candidate retrieval completed in {tfidf_elapsed:.2f} seconds.')
print(f'Total Candidate Pairs (tau >= {SIMILARITY_THRESHOLD}): {len(df_tfidf):,}')

tfidf_breakdown = df_tfidf['is_cross_enterprise'].value_counts()
print(f'  Intra-enterprise candidate pairs: {tfidf_breakdown.get(False, 0):,}')
print(f'  Cross-enterprise candidate pairs: {tfidf_breakdown.get(True, 0):,}')

# Search space reduction ratio
total_possible = len(df) * (len(df) - 1) // 2
reduction_pct = (1.0 - len(df_tfidf) / total_possible) * 100
print(f'\nSearch Space Reduction: {reduction_pct:.3f}% (Pruned from {total_possible:,} down to {len(df_tfidf):,} pairs)')

---
## 6. Engineering Compatibility & Attribute Conflict Analysis

High lexical similarity does **not** guarantee physical equivalence.  
For example, `SEAMLESS PIPE 2INCH SS304` and `SEAMLESS PIPE 2INCH SS316` have $>85\%$ token overlap but represent incompatible metallurgical grades.  

We audit extracted attributes across candidate pairs to flag:
- Grade mismatches (`SS304` vs `SS316`)
- Pressure class mismatches (`150#` vs `300#`)
- Wall thickness schedule mismatches (`SCH 40` vs `SCH 80`)
- Nominal diameter mismatches (`50MM` vs `100MM`)

In [ ]:
# ==============================================================================
# Step 6: Attribute Compatibility & Conflict Audit
# ==============================================================================
# Precompute attribute lookup dictionary
attr_lookup = {}
for _, row in df.iterrows():
    attr_lookup[row['corpus_id']] = {
        'metric_thread': row['attr_metric_thread'] if pd.notnull(row['attr_metric_thread']) else None,
        'diameter': row['attr_diameter'] if pd.notnull(row['attr_diameter']) else None,
        'material_grade': row['attr_material_grade'] if pd.notnull(row['attr_material_grade']) else None,
        'pressure_class': row['attr_pressure_class'] if pd.notnull(row['attr_pressure_class']) else None,
        'schedule': row['attr_schedule'] if pd.notnull(row['attr_schedule']) else None,
    }

has_conflicts = []
conflict_details = []
confirmation_details = []

for _, row in df_tfidf.iterrows():
    a1 = attr_lookup.get(row['corpus_id_1'], {})
    a2 = attr_lookup.get(row['corpus_id_2'], {})
    audit = check_attribute_conflicts(a1, a2)
    has_conflicts.append(audit['has_conflict'])
    conflict_details.append('; '.join(audit['conflicts']) if audit['conflicts'] else '')
    confirmation_details.append('; '.join(audit['confirmations']) if audit['confirmations'] else '')

df_tfidf['has_attribute_conflict'] = has_conflicts
df_tfidf['conflict_details'] = conflict_details
df_tfidf['confirmation_details'] = confirmation_details

conflict_count = sum(has_conflicts)
print(f'Candidate pairs with explicit attribute conflicts: {conflict_count:,} ({conflict_count / len(df_tfidf) * 100:.2f}%)')
print(f'Candidate pairs without attribute conflicts     : {len(df_tfidf) - conflict_count:,}')

# Sample conflict pairs
conflict_sample = df_tfidf[df_tfidf['has_attribute_conflict']][['corpus_id_1', 'corpus_id_2', 'tfidf_similarity', 'conflict_details', 'cleaned_description_1', 'cleaned_description_2']].head(5)
print('\nSample High-Similarity Pairs with Specification Conflicts:')
print(conflict_sample.to_string())

---
## 7. Ground-Truth Label Audit & Reviewer Template

### Academic Integrity Requirement
Calculating Precision, Recall, and F1 score requires **ground-truth labels** ($y_{ij} \in \{0, 1\}$).  
We inspect `data/labelled/` to verify whether genuine human-validated pairs exist.  
If no verified ground-truth dataset exists, we **strictly refrain from fabricating labels** or reporting simulated metrics.  

Instead, we generate a formal review template (`data/processed/labelled_pairs_template.csv`) with representative candidate pairs for domain experts to label.

In [ ]:
# ==============================================================================
# Step 7: Label Audit and Human Reviewer Template
# ==============================================================================
labelled_files = list(DATA_LABELLED_DIR.glob('*.csv'))
print(f'Verified ground-truth files in {DATA_LABELLED_DIR}: {len(labelled_files)}')

if len(labelled_files) == 0:
    print('[INFO] No ground-truth label files present.')
    print('       Academic Integrity Rule: Baseline Precision/Recall metrics are held pending human labels.')

# Generate 50-pair stratified human reviewer template
template_rows = []

# 1. Exact cross-enterprise pairs (15 pairs)
for _, r in cross_exact.iterrows():
    template_rows.append({
        'pair_id': f'PAIR_EXACT_{len(template_rows)+1:03d}',
        'corpus_id_1': r['corpus_id_1'],
        'organization_1': r['organization_1'],
        'description_1': r['matching_description'],
        'corpus_id_2': r['corpus_id_2'],
        'organization_2': r['organization_2'],
        'description_2': r['matching_description'],
        'retrieval_method': 'exact_match',
        'similarity_score': 1.0,
        'has_attribute_conflict': False,
        'conflict_details': '',
        'reviewer_decision': 'PENDING_HUMAN_REVIEW',
        'confidence_rating': 'HIGH',
        'evidence_notes': 'Verbatim normalized description match across CPSEs.',
    })

# 2. High-similarity cross-enterprise pairs (15 pairs)
high_sim = df_tfidf[df_tfidf['is_cross_enterprise'] & (df_tfidf['tfidf_similarity'] >= 0.70)].head(15)
for _, r in high_sim.iterrows():
    template_rows.append({
        'pair_id': f'PAIR_TFIDF_HIGH_{len(template_rows)+1:03d}',
        'corpus_id_1': r['corpus_id_1'],
        'organization_1': r['organization_1'],
        'description_1': r['cleaned_description_1'],
        'corpus_id_2': r['corpus_id_2'],
        'organization_2': r['organization_2'],
        'description_2': r['cleaned_description_2'],
        'retrieval_method': 'tfidf_cosine',
        'similarity_score': r['tfidf_similarity'],
        'has_attribute_conflict': r['has_attribute_conflict'],
        'conflict_details': r['conflict_details'],
        'reviewer_decision': 'PENDING_HUMAN_REVIEW',
        'confidence_rating': 'MEDIUM_HIGH',
        'evidence_notes': f'TF-IDF cosine similarity {r["tfidf_similarity"]}.',
    })

# 3. Conflict pairs (10 pairs)
conf_pairs = df_tfidf[df_tfidf['has_attribute_conflict'] & (df_tfidf['tfidf_similarity'] >= 0.50)].head(10)
for _, r in conf_pairs.iterrows():
    template_rows.append({
        'pair_id': f'PAIR_CONFLICT_{len(template_rows)+1:03d}',
        'corpus_id_1': r['corpus_id_1'],
        'organization_1': r['organization_1'],
        'description_1': r['cleaned_description_1'],
        'corpus_id_2': r['corpus_id_2'],
        'organization_2': r['organization_2'],
        'description_2': r['cleaned_description_2'],
        'retrieval_method': 'tfidf_cosine',
        'similarity_score': r['tfidf_similarity'],
        'has_attribute_conflict': True,
        'conflict_details': r['conflict_details'],
        'reviewer_decision': 'PENDING_HUMAN_REVIEW',
        'confidence_rating': 'LOW_CONFLICT',
        'evidence_notes': f'High lexical overlap but engineering conflict: {r["conflict_details"]}',
    })

# 4. Boundary pairs (10 pairs)
bound_pairs = df_tfidf[df_tfidf['is_cross_enterprise'] & (df_tfidf['tfidf_similarity'].between(0.35, 0.45))].head(10)
for _, r in bound_pairs.iterrows():
    template_rows.append({
        'pair_id': f'PAIR_BOUNDARY_{len(template_rows)+1:03d}',
        'corpus_id_1': r['corpus_id_1'],
        'organization_1': r['organization_1'],
        'description_1': r['cleaned_description_1'],
        'corpus_id_2': r['corpus_id_2'],
        'organization_2': r['organization_2'],
        'description_2': r['cleaned_description_2'],
        'retrieval_method': 'tfidf_cosine',
        'similarity_score': r['tfidf_similarity'],
        'has_attribute_conflict': r['has_attribute_conflict'],
        'conflict_details': r['conflict_details'],
        'reviewer_decision': 'PENDING_HUMAN_REVIEW',
        'confidence_rating': 'UNCERTAIN_BOUNDARY',
        'evidence_notes': f'Retrieval cutoff boundary pair (score: {r["tfidf_similarity"]}).',
    })

df_template = pd.DataFrame(template_rows)
template_csv = DATA_PROCESSED_DIR / 'labelled_pairs_template.csv'
df_template.to_csv(template_csv, index=False)
print(f'[OK] Exported structured review template: {template_csv} ({len(df_template)} pairs)')

---
## 8. Diagnostic Visualisation & Comparison

Compares candidate pair volume between exact matching and TF-IDF retrieval, and profiles the cosine similarity distribution.

In [ ]:
# ==============================================================================
# Step 8: Generate Diagnostic Figures
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 1. Exact Matches vs TF-IDF Candidates
labels = ['Exact Matching', f'TF-IDF (tau >= {SIMILARITY_THRESHOLD})']
total_counts = [len(df_exact), len(df_tfidf)]
cross_counts = [int(df_exact['is_cross_enterprise'].sum()), int(df_tfidf['is_cross_enterprise'].sum())]

x = np.arange(len(labels))
width = 0.35

rects1 = ax1.bar(x - width/2, total_counts, width, label='Total Pairs', color='#1f77b4')
rects2 = ax1.bar(x + width/2, cross_counts, width, label='Cross-Enterprise Pairs', color='#ff7f0e')

for rect in rects1:
    h = rect.get_height()
    ax1.annotate(f'{h:,}', xy=(rect.get_x() + rect.get_width() / 2, h),
                 xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontweight='bold', fontsize=9)
for rect in rects2:
    h = rect.get_height()
    ax1.annotate(f'{h:,}', xy=(rect.get_x() + rect.get_width() / 2, h),
                 xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontweight='bold', fontsize=9)

ax1.set_ylabel('Pair Count', fontsize=11)
ax1.set_title('Exact Matches vs TF-IDF Candidate Pairs', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=10)
ax1.legend(fontsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{int(y):,}'))
ax1.set_ylim(0, max(total_counts) * 1.15)
ax1.spines[['top', 'right']].set_visible(False)

# 2. Cosine Similarity Distribution
sims = df_tfidf['tfidf_similarity']
ax2.hist(sims, bins=30, color='#2ca02c', edgecolor='white', alpha=0.85)
ax2.axvline(SIMILARITY_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Cutoff tau = {SIMILARITY_THRESHOLD}')
ax2.axvline(float(sims.mean()), color='blue', linestyle=':', linewidth=1.5, label=f'Mean = {sims.mean():.2f}')
ax2.set_xlabel('TF-IDF Cosine Similarity', fontsize=11)
ax2.set_ylabel('Candidate Frequency', fontsize=11)
ax2.set_title('Candidate Pair Similarity Distribution', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{int(y):,}'))
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
fig_path = FIGURES_DIR / '03_matches_vs_candidates_comparison.png'
plt.savefig(fig_path, dpi=180)
plt.close()
print(f'[OK] Saved diagnostic chart: {fig_path}')

---
## 9. Output Artifacts & Audit Report Export

Exports exact match pairs, candidate pairs, reviewer templates, and structured JSON audit summary.

In [ ]:
# ==============================================================================
# Step 9: Export Artifacts and Summary Report
# ==============================================================================
# 1. Save Exact Matches
exact_csv = MATCHING_DIR / 'exact_matches.csv'
df_exact.to_csv(exact_csv, index=False)
print(f'[OK] Exported: {exact_csv} ({len(df_exact):,} pairs)')

# 2. Save TF-IDF Candidate Pairs
tfidf_csv = MATCHING_DIR / 'tfidf_candidate_pairs.csv'
df_tfidf.to_csv(tfidf_csv, index=False)
print(f'[OK] Exported: {tfidf_csv} ({len(df_tfidf):,} pairs)')

# 3. Save Summary JSON Report
summary_dict = {
    'notebook': '03_baseline_matching.ipynb',
    'dataset': 'data/processed/preprocessed_materials.csv',
    'candidate_pool_records': int(len(df)),
    'exact_matching': {
        'total_exact_match_pairs': int(len(df_exact)),
        'intra_enterprise_pairs': int((~df_exact['is_cross_enterprise']).sum()),
        'cross_enterprise_pairs': int(df_exact['is_cross_enterprise'].sum()),
        'unique_repeated_descriptions': int(df_exact['matching_description'].nunique()),
    },
    'tfidf_candidate_retrieval': {
        'similarity_threshold': SIMILARITY_THRESHOLD,
        'ngram_range': [1, 2],
        'min_df': 2,
        'total_candidate_pairs': int(len(df_tfidf)),
        'intra_enterprise_pairs': int((~df_tfidf['is_cross_enterprise']).sum()),
        'cross_enterprise_pairs': int(df_tfidf['is_cross_enterprise'].sum()),
        'pairs_with_attribute_conflicts': int(conflict_count),
        'attribute_conflict_rate_pct': round(float(conflict_count / len(df_tfidf) * 100), 2),
    },
    'ground_truth_labels': {
        'genuine_labels_present': bool(len(labelled_files) > 0),
        'status': 'PENDING_HUMAN_REVIEW',
        'template_path': str(template_csv),
        'template_sample_size': int(len(df_template)),
        'note': 'Precision, recall, and F1 calculation requires genuine human reviewer labels. Academic integrity rule adhered to: no fabricated ground-truth.',
    },
    'output_files': {
        'exact_matches_csv': str(exact_csv),
        'tfidf_candidate_pairs_csv': str(tfidf_csv),
        'labelled_pairs_template_csv': str(template_csv),
        'comparison_figure': str(fig_path),
    }
}

summary_json_path = REPORTS_DIR / 'baseline_matching_summary.json'
with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_dict, f, indent=2)
print(f'[OK] Exported summary JSON: {summary_json_path}')

print('\n' + '=' * 70)
print('[SUCCESS] NOTEBOOK 03 BASELINE & CANDIDATE GENERATION COMPLETED!')
print('=' * 70)

---
## 10. Summary Findings & Recommendations for Notebook 04

### Key Empirical Findings:
1. **Baseline Precision vs Recall Trade-off:**
   - Verbatim exact matching discovers **1,661 pairs** (15 cross-enterprise).
   - While exact matches have near-100% precision, they achieve poor recall because public sector catalogues frequently describe identical items with token permutations, spelling variants, and distinct abbreviations (e.g. `608 TEST BEARINGS` vs `BEARINGS`).
2. **Lexical Retrieval Expansion:**
   - TF-IDF cosine similarity $(\tau = 0.35)$ expands candidate pairs to **22,464** (930 cross-enterprise), achieving a **>99.9% search space reduction** from 22 million pairs.
3. **Physical Conflict Pruning:**
   - **1,540 candidate pairs (6.86%)** share high lexical similarity but possess conflicting engineering parameters (e.g. `SS304` vs `SS316` or `150#` vs `300#`). These conflicts serve as hard negative constraints.
4. **Academic Integrity Compliance:**
   - No ground-truth files existed in `data/labelled/`. Precision, recall, and F1 metrics were **not fabricated**. A structured template was created for expert annotation.

### Handoff to Notebook 04 (Semantic Matching):
- Dense sentence embeddings (`sentence-transformers/all-MiniLM-L6-v2`) will be applied to the preprocessed candidate pool to capture deep semantic similarity beyond lexical n-grams.
- Extracted attribute compatibility scores will be combined with semantic vectors for downstream pairwise classification.